## 面试问题

超大 observation 怎么做选择性读取与分页？

## 回答主线

单个 observation 很大时，既不能整个塞入（爆窗口）也不能盲目截断前 N 行（关键信息可能在后面）。做法是把读取变成循环内的动作：先给页级摘要，让模型选择性读取命中页。本 Notebook 用一段 ERROR 在第 42 行的 60 行日志，对比截断前 20 行（漏读）与分页选择性读取（定位成功）。

## 真实案例

60 行服务日志，`ERROR disk full` 在第 42 行。任务是定位错误。按每 20 行分页，先看页级摘要再读命中页。数据为教学日志，不代表真实日志系统。

In [1]:
log_lines = []  # 构造一段教学日志。
for i in range(1, 61):  # 生成 60 行日志。
    if i == 42:  # 在第 42 行埋入错误。
        log_lines.append(f"line {i}: ERROR disk full")  # 关键错误行。
    else:  # 其余为普通信息行。
        log_lines.append(f"line {i}: INFO ok")  # 普通信息行。

print("日志总行数:", len(log_lines))  # 展示日志规模。
print("首行:", log_lines[0])  # 展示日志开头。
print("含 ERROR 的真实行号:", [i + 1 for i, l in enumerate(log_lines) if "ERROR" in l])  # 展示错误真实位置。

日志总行数: 60
首行: line 1: INFO ok
含 ERROR 的真实行号: [42]


## 基线（Baseline）

反面基线：截断只读前 20 行。因为 ERROR 在第 42 行，截断窗口里没有它，策略会错误地得出「无错误」。

In [2]:
def read_truncated(lines, limit=20):  # 截断策略：只读前 limit 行。
    window = lines[:limit]  # 取开头窗口。
    found = any("ERROR" in l for l in window)  # 在窗口内查找错误。
    return window, found  # 返回窗口与是否命中。

trunc_window, trunc_found = read_truncated(log_lines)  # 截断读取前 20 行。
print("截断读取行数:", len(trunc_window))  # 展示只读了前 20 行。
print("截断策略是否发现 ERROR:", trunc_found)  # 展示截断漏掉第 42 行错误。

截断读取行数: 20
截断策略是否发现 ERROR: False


## 失败案例与修正

截断的危害是「静默漏读」——模型不知道后面还有内容，会基于不完整信息自信作答。修正是分页 + 页级摘要 + 选择性读取：先看每页是否命中，只读命中页，既省上下文又不漏关键行。

In [3]:
def paginate(lines, page_size=20):  # 把日志分页。
    pages = []  # 收集分页。
    for start in range(0, len(lines), page_size):  # 按页大小切分。
        pages.append(lines[start:start + page_size])  # 追加一页。
    return pages  # 返回所有页。

def page_summaries(pages):  # 为每页生成命中摘要。
    summaries = []  # 收集页级摘要。
    for idx, page in enumerate(pages):  # 遍历每一页。
        has_error = any("ERROR" in l for l in page)  # 判断该页是否含错误。
        summaries.append({"page": idx, "has_error": has_error, "lines": len(page)})  # 记录页级信号。
    return summaries  # 返回页级摘要。

pages = paginate(log_lines)  # 对日志分页。
summaries = page_summaries(pages)  # 生成页级摘要。
print("总页数:", len(pages))  # 展示分页数量。
for s in summaries:  # 逐页打印摘要。
    print("  页摘要:", s)  # 展示每页是否含错误。

总页数: 3
  页摘要: {'page': 0, 'has_error': False, 'lines': 20}
  页摘要: {'page': 1, 'has_error': False, 'lines': 20}
  页摘要: {'page': 2, 'has_error': True, 'lines': 20}


In [4]:
def select_and_read(pages, summaries):  # 依据摘要选择性读取命中页。
    target = None  # 预置目标页。
    for s in summaries:  # 遍历页级摘要。
        if s["has_error"]:  # 找到含错误的页。
            target = s["page"]  # 记录目标页号。
            break  # 命中即停止选择。
    if target is None:  # 没有任何页命中。
        return None, []  # 返回未找到。
    page = pages[target]  # 只读取命中页。
    error_lines = [l for l in page if "ERROR" in l]  # 在该页定位错误行。
    return target, error_lines  # 返回页号与错误行。

hit_page, error_lines = select_and_read(pages, summaries)  # 选择性读取定位错误。
print("命中页号:", hit_page)  # 展示只读了含错误的页。
print("定位到的错误行:", error_lines)  # 展示成功定位第 42 行错误。

命中页号: 2
定位到的错误行: ['line 42: ERROR disk full']


## 结果解读

截断策略漏掉第 42 行、错误地报告无错误；分页选择性读取从页级摘要看到第 3 页（索引 2）命中，只读该页就定位到错误，且只读了 20 行而非全部 60 行。这就是「读取即动作」的价值：省上下文又不静默漏读。

In [5]:
print("截断策略结论(是否发现ERROR):", trunc_found)  # 截断策略错误地报告无错误。
print("选择性读取是否定位到ERROR:", len(error_lines) > 0)  # 选择性读取正确定位。
print("选择性读取只读", len(pages[hit_page]), "行而非全部", len(log_lines), "行")  # 展示省上下文又不漏读。

截断策略结论(是否发现ERROR): False
选择性读取是否定位到ERROR: True
选择性读取只读 20 行而非全部 60 行


In [6]:
assert trunc_found is False  # 截断策略漏掉第 42 行错误。
assert hit_page == 2  # 第 42 行落在第 3 页(索引 2)。
assert len(error_lines) == 1  # 选择性读取定位到一条错误。
assert "ERROR" in error_lines[0]  # 定位到的确实是错误行。
assert len(pages[hit_page]) < len(log_lines)  # 只读一页而非全部日志。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
